### Select mediating ontology

In [ ]:
from modules.mediating_selector import MediatingOntologySelector

o1_path = "data/anatomy/human-mouse/human.owl"
o2_path = "data/anatomy/human-mouse/mouse.owl"

def load_logmap_anchors(path: str):
    mappings = []

    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 2:
                continue
            iri1, iri2 = parts[0], parts[1]
            mappings.append((iri1, iri2))

    return mappings

lexical_mappings = load_logmap_anchors("output/logmap_anchors.tsv")

selector = MediatingOntologySelector(
    o1_path=o1_path,
    o2_path=o2_path,
    lexical_mappings=lexical_mappings
)

mediators = selector.select_top_mediators(top_k=3, download=True)

print(mediators)

### Perform LogMapBio algorithm to produce compose mappings

In [ ]:
from modules.logmap_bio_compose_runner import LogMapBioRunner

o1_path = "data/anatomy/human-mouse/human.owl"
o2_path = "data/anatomy/human-mouse/mouse.owl"

mo_path = "output/mediators/UBERON.owl"

runner = LogMapBioRunner(
    o1_path=o1_path,
    o2_path=o2_path,
    mediating_ontology_path=mo_path,
    work_dir="output/logmapbio"
)

# Run composition only (step 3)
# results_compose = runner.run_composition_only()

# Run full pipeline (MC - Direct)
results_full = runner.run()

print("=== Full MC Pipeline ===")
print(f"M1 output: {results_full['M1_path']}")
print(f"M2 output: {results_full['M2_path']}")
print(f"MC composed mappings: {results_full['MC_file']}")
print(f"Direct LogMap mappings: {results_full['Direct_file']}")
print(f"MC minus direct mappings: {results_full['MC_minus_direct_file']}")
print(f"MC size: {results_full['MC_size']}")
print(f"Direct size: {results_full['Direct_size']}")
print(f"MC - Direct size: {results_full['MC_minus_size']}")

### Compare mediator mappings aganst logmap

In [ ]:
from modules.evaluator import OntologyAlignmentEvaluator
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

DATASET_NAME = "anatomy"
GT_PATH = "data/anatomy/human-mouse/refs_equiv/full.tsv"
OUTPUT_DIR = "results_mediator"

LOGMAP_FILE = "output/logmapbio/direct_mappings.txt"

OTHER_SYSTEMS = {
    "MC_minus_direct": "output/logmapbio/MC_minus_direct.txt",
    "MC_composed": "output/logmapbio/MC_composed.txt",
}

PRED_COL = "Prediction"

def load_logmap_file(file_path: str) -> pd.DataFrame:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    records = []
    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            src, tgt = parts[0], parts[1]
            score = float(parts[3]) if len(parts) > 3 else 1.0
            records.append({
                "Source": src,
                "Target": tgt,
                "LogMapScore": score,         
                "Prediction": score > 0       
            })

    return pd.DataFrame(records)

print("\nInitializing OntologyAlignmentEvaluator...")
evaluator = OntologyAlignmentEvaluator(GT_PATH)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
results_path = Path(OUTPUT_DIR) / timestamp / DATASET_NAME
results_path.mkdir(parents=True, exist_ok=True)

summary_rows = []

print("\nEvaluating LogMap...")
logmap_df = load_logmap_file(LOGMAP_FILE)
logmap_metrics = evaluator.evaluate(
    df=logmap_df,
    dataset_name=DATASET_NAME,
    experiment_type="logmap",
    prompts_used="logmap",
    second_system_name="LogMap",
    second_system_pred_col=PRED_COL,
    results_dir=OUTPUT_DIR
)["LogMap"]

summary_rows.append({"System": "LogMap", **{k: v for k, v in logmap_metrics.items() if k != "ConfusionMatrix"}})

for name, path in OTHER_SYSTEMS.items():
    print(f"\nEvaluating {name}...")
    df = load_logmap_file(path)
    metrics = evaluator.evaluate(
        df=df,
        dataset_name=DATASET_NAME,
        experiment_type=name,
        prompts_used="logmap_mediator",
        second_system_name=name,
        second_system_pred_col=PRED_COL,
        results_dir=OUTPUT_DIR
    )[name]

    summary_rows.append({"System": name, **{k: v for k, v in metrics.items() if k != "ConfusionMatrix"}})

metrics_df = pd.DataFrame(summary_rows)

columns_order = [
    "System", "Precision", "Recall", "F1", "TP", "TN", "FP", "FN",
    "Sensitivity", "Specificity", "YoudenIndex"
]
metrics_df = metrics_df[columns_order]

metrics_csv = results_path / "metrics_comparison.csv"
metrics_df.to_csv(metrics_csv, index=False)

print("\n=== FINAL METRICS ===")
print(metrics_df.to_string(index=False))

best_method = metrics_df.sort_values(by="F1", ascending=False).iloc[0]["System"]
print(f"\nBest method based on F1: –> {best_method}")

### Validate mediator mappings using LLM

In [ ]:
from utils.onto_object import OntologyEntryAttr, OntologyAccess
from modules.llm_validator import LLMValidator
from modules.evaluator import OntologyAlignmentEvaluator
import pandas as pd
from datetime import datetime
import tqdm

DATASET_NAME = "anatomy"
ONTO_SRC_PATH = "data/anatomy/human-mouse/human.owl"
ONTO_TGT_PATH = "data/anatomy/human-mouse/mouse.owl"
GT_PATH = "/Users/shuma/Desktop/dyplom/data/anatomy/human-mouse/refs_equiv/full.tsv"
INPUT_FILE = "output/logmapbio/MC_minus_direct.txt"
OUTPUT_DIR = "results"

BATCH_SIZE = None

onto_src = OntologyAccess(ONTO_SRC_PATH, annotate_on_init=True)
onto_tgt = OntologyAccess(ONTO_TGT_PATH, annotate_on_init=True)
validator = LLMValidator()
evaluator = OntologyAlignmentEvaluator(GT_PATH)

with open(INPUT_FILE, "r") as f:
    lines = f.readlines()

if BATCH_SIZE:
    lines = lines[:BATCH_SIZE]

print(f"Total pairs to validate: {len(lines)}")
results = []

# VALIDATE WITH LLM
for line in tqdm.tqdm(lines):
    parts = line.strip().split("|")
    if len(parts) < 2:
        continue

    src_uri, tgt_uri = parts[0], parts[1]
    logmap_score = float(parts[3]) if len(parts) > 3 else 1.0

    src_entity = OntologyEntryAttr(class_uri=src_uri, onto=onto_src)
    tgt_entity = OntologyEntryAttr(class_uri=tgt_uri, onto=onto_tgt)

    res = validator.validate(
        src_entity,
        tgt_entity,
        prompt_type="sequential_hierarchy_with_synonyms",
        system_prompt_type="synonym_aware"
    )

    results.append({
        "Source": src_uri,
        "Target": tgt_uri,
        "LLMDecision": res["is_match"],
        "LLMConfidence": res["confidence"],
        "LogMapScore": logmap_score,
    })

    print(f"Validated pair: {src_uri} -> {tgt_uri}, Result: {res}")

df = pd.DataFrame(results)

# EVALUATE
experiment_type = "sequential_hierarchy_with_synonyms"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

report = evaluator.evaluate(
    df=df,
    dataset_name=DATASET_NAME,
    experiment_type=experiment_type,
    prompts_used=experiment_type,
    results_dir=OUTPUT_DIR
)

print("\n=== LLM Evaluation Report ===")
print(report)

# SAMPLE DETAILED RESULTS
detailed_results_path = f"{OUTPUT_DIR}/{timestamp}/{DATASET_NAME}/{experiment_type}/detailed_results.csv"
print("\n=== Sample of Detailed Results ===")
print(pd.read_csv(detailed_results_path).head())